# ADHD-200 cleaning — **multi-class** version (Control / ADHD-Combined / ADHD-Inattentive)

**Input:** `adhd200_preprocessed_phenotypics.tsv`  →  **Output:** `adhd_cleaned_multiclass.csv`

Target `DX` (3 classes):

| DX | Meaning |
|---:|---|
| 0 | Control (typically developing) |
| 1 | ADHD-Combined (+ the very rare Hyperactive/Impulsive subtype, merged) |
| 2 | ADHD-Inattentive |

Steps: load → unify missing-value tokens → fix types → standardise Handedness → drop unusable columns → build 3-class target → complete-case filter → sanity checks → save.

In [1]:
import numpy as np
import pandas as pd

RAW_FILE = 'adhd200_preprocessed_phenotypics.tsv'
OUT_FILE = 'adhd_cleaned_multiclass.csv'

## 1. Load

Everything is read as text first so no value is silently altered by type inference. `pending`, `N/A` and `-999` all mean "missing" in this dataset, so they are converted to `NaN` in one place.

In [2]:
df = pd.read_csv(RAW_FILE, sep='\t', dtype=str)
df.columns = df.columns.str.strip().str.replace(r'[ /]', '_', regex=True)   # 'Hyper/Impulsive' -> 'Hyper_Impulsive'
df = df.apply(lambda s: s.str.strip())

MISSING_TOKENS = r'(?i)^(pending|n/?a|nan|-999(\.0+)?)$'
df = df.replace(MISSING_TOKENS, np.nan, regex=True)

print('raw shape:', df.shape)
df.head()

raw shape: (973, 19)


,ScanDir_ID,Site,Gender,Age,Handedness,DX,Secondary_Dx,ADHD_Measure,ADHD_Index,Inattentive,Hyper_Impulsive,IQ_Measure,Verbal_IQ,Performance_IQ,Full2_IQ,Full4_IQ,Med_Status,QC_Athena,QC_NIAK
0,2371032,3,0,10.73,1,0,NaN,2,47,55,43,1,121,119,NaN,122,1,1,1
1,2026113,3,0,12.99,1,1,NaN,2,90,89,78,1,122,108,NaN,106,1,1,1
2,3434578,3,0,8.12,1,0,NaN,2,42,42,43,1,85,98,NaN,89,1,1,1
3,8628223,3,0,10.81,1,0,Simple phobia,2,42,49,49,1,85,86,NaN,97,1,1,1
4,1623716,3,0,12.65,1,1,NaN,2,87,90,90,1,89,88,NaN,89,1,1,1


In [3]:
df["Site"].value_counts()

Site
5    263
1    245
6    113
7     98
3     94
4     73
8     61
2     26
Name: count, dtype: int64

In [5]:
df["IQ_Measure"].value_counts()

IQ_Measure
2    473
3    245
1     94
4     61
5     25
Name: count, dtype: int64

### ADHD-200 imaging sites

| Site ID | Imaging Center | Samples |
|---:|---|---:|
| 1 | Peking University | 245 |
| 2 | Bradley Hospital / Brown University | 26 |
| 3 | Kennedy Krieger Institute (KKI) | 94 |
| 4 | NeuroIMAGE | 73 |
| 5 | New York University (NYU) | 263 |
| 6 | Oregon Health & Science University (OHSU) | 113 |
| 7 | University of Pittsburgh | 98 |
| 8 | Washington University in St. Louis | 61 |
| | **Total** | **973** |

Site 2 (Brown) has no public labels (`DX = pending`), so it disappears at the target step.

## 2. Fix types

All columns except `Secondary_Dx` are numeric. One Peking row stores handedness as the letter `L`; this is mapped to the left-handed code (`0`) instead of being thrown away.

In [3]:
df['Handedness'] = df['Handedness'].str.upper().replace({'L': '0', 'R': '1'})

NUMERIC_COLS = [c for c in df.columns if c != 'Secondary_Dx']
df[NUMERIC_COLS] = df[NUMERIC_COLS].apply(pd.to_numeric, errors='coerce')

df.dtypes

ScanDir_ID           int64
Site                 int64
Gender             float64
Age                float64
Handedness         float64
DX                 float64
Secondary_Dx        object
ADHD_Measure       float64
ADHD_Index         float64
Inattentive        float64
Hyper_Impulsive    float64
IQ_Measure         float64
Verbal_IQ          float64
Performance_IQ     float64
Full2_IQ           float64
Full4_IQ           float64
Med_Status         float64
QC_Athena          float64
QC_NIAK            float64
dtype: object

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 973 entries, 0 to 972
Data columns (total 19 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   ScanDir_ID       973 non-null    object
 1   Site             973 non-null    object
 2   Gender           972 non-null    object
 3   Age              973 non-null    object
 4   Handedness       964 non-null    object
 5   DX               947 non-null    object
 6   Secondary_Dx     131 non-null    object
 7   ADHD_Measure     629 non-null    object
 8   ADHD_Index       557 non-null    object
 9   Inattentive      636 non-null    object
 10  Hyper_Impulsive  636 non-null    object
 11  IQ_Measure       898 non-null    object
 12  Verbal_IQ        672 non-null    object
 13  Performance_IQ   672 non-null    object
 14  Full2_IQ         71 non-null     object
 15  Full4_IQ         887 non-null    object
 16  Med_Status       564 non-null    object
 17  QC_Athena        965 non-null    ob

## 3. Handedness

Handedness was not recorded the same way at every site:

- **Sites 1–4, 6–8:** categorical — `0` = left, `1` = right, `2` = ambidextrous.
- **NYU (site 5):** continuous Edinburgh Handedness Inventory score in `[-1, 1]` (negative = left, positive = right, `0` = no preference).
- **NeuroIMAGE (site 4):** one participant has the value `3`; the dataset maintainers say this is a recoding error and it should be `2`.

All values are converted to the single categorical coding `0 / 1 / 2`. Anything that fits none of the rules becomes missing.

In [4]:
site, hand = df['Site'], df['Handedness']

is_nyu    = site.eq(5)
cat_ok    = ~is_nyu & hand.isin([0, 1, 2])      # categorical coding (all sites except NYU)
fix_site4 = site.eq(4) & hand.eq(3)             # NeuroIMAGE recoding error: 3 -> 2
ehi_ok    = is_nyu & hand.between(-1, 1)        # NYU Edinburgh score

hand_std = pd.Series(np.nan, index=df.index, dtype=float)
hand_std.loc[cat_ok]    = hand.loc[cat_ok]
hand_std.loc[fix_site4] = 2
hand_std.loc[ehi_ok]    = np.select([hand[ehi_ok] < 0, hand[ehi_ok] > 0], [0, 1], default=2)

invalid = hand.notna() & hand_std.isna()
print('Handedness values that matched no rule (set to missing):', int(invalid.sum()))

df['Handedness'] = hand_std
print(pd.crosstab(df['Site'], df['Handedness'].fillna(-1)).rename(columns={-1.0: 'missing'}))

Handedness values that matched no rule (set to missing): 0
Handedness  missing  0.0  1.0  2.0
Site                              
1                 0    4  241    0
2                 0    1   25    0
3                 0    9   84    1
4                 2    8   62    1
5                 7    5  250    1
6                 0    0  113    0
7                 0    4   94    0
8                 0    0   61    0


## 4. Drop columns that should not be features

| Column | Reason |
|---|---|
| `ScanDir_ID` | identifier |
| `Site` | only needed for the handedness fix above; keeping it lets a model learn the site instead of ADHD |
| `QC_Athena`, `QC_NIAK` | neuroimaging quality flags, not phenotypic features |
| `Full2_IQ` | ~92 % missing |
| `Secondary_Dx` | free-text comorbid diagnosis, mostly empty and recorded after diagnosis |
| `Med_Status` | ~40 % missing, and the missingness depends on site and diagnosis; medication is also a consequence of diagnosis. Keeping it removes many otherwise-usable rows in the complete-case step below |

In [5]:
DROP_COLS = ['ScanDir_ID', 'Site', 'QC_Athena', 'QC_NIAK', 'Full2_IQ', 'Secondary_Dx', 'Med_Status']
df = df.drop(columns=DROP_COLS)
print('remaining columns:', list(df.columns))

remaining columns: ['Gender', 'Age', 'Handedness', 'DX', 'ADHD_Measure', 'ADHD_Index', 'Inattentive', 'Hyper_Impulsive', 'IQ_Measure', 'Verbal_IQ', 'Performance_IQ', 'Full4_IQ']


## 5. Multi-class target

ADHD-200 codes `DX` as `0` Control, `1` ADHD-Combined, `2` ADHD-Hyperactive/Impulsive, `3` ADHD-Inattentive.

- Rows with unknown `DX` (`pending`, the held-out test labels) cannot be used for supervised learning and are dropped.
- The Hyperactive/Impulsive subtype is extremely rare (only a handful of rows survive cleaning), far too few to learn from or to stratify across CV folds. It is merged into ADHD-Combined, the clinically closest group (both show hyperactivity/impulsivity).
- The remaining labels are re-coded to consecutive integers `0, 1, 2`, which is what scikit-learn and XGBoost expect (XGBoost fails on a gap such as `0, 1, 3`).

The merge is done on the original codes, so the class sizes printed below are the ones that will be modelled.

In [6]:
n_raw = len(df)
df = df.dropna(subset=['DX']).copy()
print(f'dropped {n_raw - len(df)} rows with unknown DX')

assert df['DX'].isin([0, 1, 2, 3]).all(), 'unexpected DX value'
print('original DX counts:', df['DX'].value_counts().sort_index().to_dict())

DX_MAP = {0: 0,      # Control
          1: 1,      # ADHD-Combined
          2: 1,      # Hyperactive/Impulsive -> merged into Combined
          3: 2}      # ADHD-Inattentive
df['DX'] = df['DX'].astype(int).map(DX_MAP)
print('3-class DX counts :', df['DX'].value_counts().sort_index().to_dict())

dropped 26 rows with unknown DX
original DX counts: {0.0: 585, 1.0: 212, 2.0: 13, 3.0: 137}
3-class DX counts : {0: 585, 1: 225, 2: 137}


## 6. Missing values

The rating scales (`ADHD_Index`, `Inattentive`, `Hyper_Impulsive`) and `Verbal_IQ` / `Performance_IQ` were never recorded at some sites, so those values are missing for whole sites rather than at random. Imputing them would invent entire columns, so rows with any missing value are removed (complete-case analysis).

In [7]:
missing = pd.DataFrame({
    'missing': df.isna().sum(),
    'percent': (df.isna().mean() * 100).round(2),
}).sort_values('percent', ascending=False)
print(missing)

print('\nshape before:', df.shape)
df = df.dropna().reset_index(drop=True)
print('shape after :', df.shape)
print('DX counts   :', df['DX'].value_counts().sort_index().to_dict())

                 missing  percent
ADHD_Index           390    41.18
ADHD_Measure         318    33.58
Inattentive          311    32.84
Hyper_Impulsive      311    32.84
Verbal_IQ            301    31.78
Performance_IQ       301    31.78
Full4_IQ              86     9.08
IQ_Measure            49     5.17
Handedness             9     0.95
Gender                 1     0.11
Age                    0     0.00
DX                     0     0.00

shape before: (947, 12)
shape after : (505, 12)
DX counts   : {0: 271, 1: 130, 2: 104}


## 7. Final types, column order and sanity checks

In [8]:
CODE_COLS = ['Gender', 'Handedness', 'ADHD_Measure', 'IQ_Measure', 'DX']
df[CODE_COLS] = df[CODE_COLS].astype(int)

df = df[[c for c in df.columns if c != 'DX'] + ['DX']]      # target last

assert df.isna().sum().sum() == 0
assert set(df['DX'].unique()) == {0, 1, 2}
assert df['Handedness'].isin([0, 1, 2]).all()
assert not df.eq(-999).any().any()
assert df['DX'].value_counts().min() >= 30, 'a class is too small for stratified CV'

print('duplicate rows:', int(df.duplicated().sum()))
print(df.dtypes.to_string())
df.describe().T

duplicate rows: 0
Gender               int64
Age                float64
Handedness           int64
ADHD_Measure         int64
ADHD_Index         float64
Inattentive        float64
Hyper_Impulsive    float64
IQ_Measure           int64
Verbal_IQ          float64
Performance_IQ     float64
Full4_IQ           float64
DX                   int64


,count,mean,std,min,25%,50%,75%,max
Gender,505.0,0.663366,0.473027,0.00,0.0,1.00,1.0,1.00
Age,505.0,11.350436,2.349518,7.17,9.5,11.09,13.0,17.96
Handedness,505.0,0.976238,0.176579,0.00,1.0,1.00,1.0,2.00
ADHD_Measure,505.0,1.562376,0.496586,1.00,1.0,2.00,2.0,2.00
ADHD_Index,505.0,49.529703,17.157970,18.00,40.0,46.00,62.0,99.00
Inattentive,505.0,41.761386,21.948872,9.00,23.0,42.00,59.0,90.00
Hyper_Impulsive,505.0,40.000000,23.052971,9.00,17.0,43.00,56.0,90.00
IQ_Measure,505.0,2.279208,0.720493,1.00,2.0,2.00,3.0,3.00
Verbal_IQ,505.0,112.655446,15.068893,65.00,102.0,112.00,123.0,158.00
Performance_IQ,505.0,105.912871,14.596434,54.00,96.0,106.00,117.0,139.00


## 8. Save

| DX | Class | Description |
|---:|---|---|
| 0 | Control | Typically developing |
| 1 | ADHD-Combined | Inattention + hyperactivity/impulsivity (includes the merged Hyperactive/Impulsive group) |
| 2 | ADHD-Inattentive | Primarily inattention |

These are the labels provided by ADHD-200, not model predictions.

In [9]:
df.to_csv(OUT_FILE, index=False)
print(f'saved {OUT_FILE}: {df.shape}')
print(df['DX'].value_counts().sort_index().rename({0: 'Control', 1: 'ADHD-Combined', 2: 'ADHD-Inattentive'}))

saved adhd_cleaned_multiclass.csv: (505, 12)
DX
Control             271
ADHD-Combined       130
ADHD-Inattentive    104
Name: count, dtype: int64
